# Projeto SBD — Etapa 1 | Possibilidades de análise e evidências da seção 7

**UFPB · Sistemas de Banco de Dados · Entrega: 25/11/2026**  
**Equipe:** Ana Beatriz · Maria Clara · Ayra  
**Responsável por este notebook:** Ayra (possibilidades de análise)  
**Fonte:** [Brazilian E-Commerce Public Dataset by Olist (Kaggle)](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce)

## Objetivo
Verificar, com os arquivos locais, se a base consegue responder às cinco perguntas principais da **seção 7** do documento da Etapa 1, e registrar a **evidência inicial** de cada uma. São resultados exploratórios, para indicar viabilidade. As regras de cálculo definitivas e a modelagem relacional ficam para as próximas etapas.

Este notebook complementa `Exploracao_Inicial_Olist_Etapa1.ipynb` e usa a mesma organização do repositório: os CSVs ficam em `data/raw/` e **não são versionados**.

| # | Pergunta | Eixo | Arquivos usados |
|---|---|---|---|
| P1 | Quais categorias de produtos geram mais receita? | Comercial | pedidos, itens, produtos, tradução de categorias |
| P2 | Quais vendedores apresentam maior volume de vendas? | Comercial | pedidos, itens, vendedores |
| P3 | Como os pedidos se distribuem por estado? | Comercial | pedidos, clientes |
| P4 | Qual é o tempo médio de entrega dos pedidos? | Logístico | pedidos (e clientes, por estado) |
| P5 | Existe relação entre atrasos nas entregas e avaliações? | Satisfação | pedidos, avaliações |

## 1. Preparação e leitura dos arquivos
A base deve estar em `data/raw/`. O notebook localiza a raiz do projeto tanto quando é executado a partir de `notebooks/` quanto da pasta principal.

**Dependências:** `pandas`, `ipykernel` e `IPython` (`pip install pandas ipykernel`). A leitura usa `dtype=str` para preservar os zeros iniciais dos CEPs; as conversões para data e número são feitas só nas colunas necessárias ao cálculo. A geolocalização e os pagamentos não são usados nestas perguntas.

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

pd.set_option("display.max_colwidth", 100)
pd.set_option("display.max_rows", 100)

ARQUIVOS = {
    "clientes": "olist_customers_dataset.csv",
    "pedidos": "olist_orders_dataset.csv",
    "itens": "olist_order_items_dataset.csv",
    "produtos": "olist_products_dataset.csv",
    "vendedores": "olist_sellers_dataset.csv",
    "avaliacoes": "olist_order_reviews_dataset.csv",
    "traducao_categorias": "product_category_name_translation.csv",
}

def encontrar_raiz_projeto():
    atual = Path.cwd().resolve()
    for candidata in [atual, *atual.parents]:
        pasta_csv = candidata / "data" / "raw"
        if pasta_csv.is_dir() and (candidata / "notebooks").is_dir():
            return candidata
    raise FileNotFoundError(
        "Não localizei a estrutura do repositório. Abra a pasta gestao-e-analise-ecommerce "
        "no VS Code e mantenha os CSVs dentro de data/raw/."
    )

RAIZ_PROJETO = encontrar_raiz_projeto()
pasta = RAIZ_PROJETO / "data" / "raw"

faltando = [nome for nome in ARQUIVOS.values() if not (pasta / nome).is_file()]
if faltando:
    raise FileNotFoundError("Arquivos não encontrados em data/raw/: " + ", ".join(faltando))

def ler(apelido, colunas=None):
    return pd.read_csv(pasta / ARQUIVOS[apelido], usecols=colunas, dtype=str, encoding="utf-8")

clientes = ler("clientes", ["customer_id", "customer_unique_id", "customer_state"])
pedidos = ler("pedidos")
itens = ler("itens", ["order_id", "order_item_id", "product_id", "seller_id", "price", "freight_value"])
produtos = ler("produtos", ["product_id", "product_category_name"])
vendedores = ler("vendedores", ["seller_id", "seller_city", "seller_state"])
avaliacoes = ler("avaliacoes", ["review_id", "order_id", "review_score"])
traducao = ler("traducao_categorias")

# Conversões usadas apenas nos cálculos (os tipos SQL finais serão definidos na modelagem)
colunas_data = ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
                "order_delivered_customer_date", "order_estimated_delivery_date"]
for coluna in colunas_data:
    pedidos[coluna] = pd.to_datetime(pedidos[coluna])
itens["price"] = pd.to_numeric(itens["price"])
itens["freight_value"] = pd.to_numeric(itens["freight_value"])
avaliacoes["review_score"] = pd.to_numeric(avaliacoes["review_score"])

print("Raiz do repositório:", RAIZ_PROJETO)
display(pd.DataFrame({"tabela": ["clientes", "pedidos", "itens", "produtos", "vendedores", "avaliacoes", "traducao"],
                      "registros": [len(clientes), len(pedidos), len(itens), len(produtos),
                                    len(vendedores), len(avaliacoes), len(traducao)]}))

Raiz do repositório: C:\Users\Lenovo\Downloads\analisededados\gestao-e-analise-ecommerce


,tabela,registros
0,clientes,99441
1,pedidos,99441
2,itens,112650
3,produtos,32951
4,vendedores,3095
5,avaliacoes,99224
6,traducao,71


## 2. Regras de cálculo adotadas (provisórias)
- **Receita** = soma de `price` dos itens, **sem o frete** (`freight_value`).
- Pedidos com status `canceled` ou `unavailable` ficam fora da receita e do ranking de vendedores.
- **Tempo de entrega** e **atraso** consideram só pedidos `delivered` que têm data de entrega ao cliente.
- **Atraso** = entrega ao cliente depois da data estimada de entrega.
- Como pedidos podem ter mais de uma avaliação, a nota é a **média por pedido** antes de comparar grupos.
- Antes de cada junção entre tabelas, conferimos se o número de linhas se mantém: junções ingênuas podem duplicar valores.

Estas regras são uma proposta para validação da equipe, não a definição final.

In [2]:
STATUS_EXCLUIDOS = ["canceled", "unavailable"]
SEGUNDOS_POR_DIA = 86_400

def br(valor, casas=2):
    # formata número no padrão brasileiro: 1.234,56
    return f"{valor:,.{casas}f}".replace(",", "X").replace(".", ",").replace("X", ".")

print("Status dos pedidos no arquivo:")
display(pedidos["order_status"].value_counts().rename_axis("status").reset_index(name="pedidos"))
print("Status excluídos da receita:", STATUS_EXCLUIDOS)

Status dos pedidos no arquivo:


,status,pedidos
0,delivered,96478
1,shipped,1107
2,canceled,625
3,unavailable,609
4,invoiced,314
5,processing,301
6,created,5
7,approved,2


Status excluídos da receita: ['canceled', 'unavailable']


## 3. P1 — Quais categorias de produtos geram mais receita?
**Arquivos:** pedidos, itens, produtos e tradução de categorias.  
Os itens são ligados ao pedido (para filtrar o status), ao produto (para obter a categoria) e à tradução (nome em inglês). Produtos sem categoria entram como `sem_categoria`.

In [3]:
n_itens = len(itens)
base = (itens.merge(pedidos[["order_id", "order_status"]], on="order_id", how="left")
             .merge(produtos, on="product_id", how="left")
             .merge(traducao, on="product_category_name", how="left"))
assert len(base) == n_itens, "A junção duplicou ou perdeu linhas de itens"

base = base[~base["order_status"].isin(STATUS_EXCLUIDOS)].copy()
base["categoria"] = (base["product_category_name_english"]
                     .fillna(base["product_category_name"]).fillna("sem_categoria"))

receita_categoria = (base.groupby("categoria")
                         .agg(pedidos=("order_id", "nunique"),
                              itens_vendidos=("order_id", "size"),
                              receita=("price", "sum"))
                         .round(2).sort_values("receita", ascending=False).reset_index())

print(f"Itens considerados: {len(base):,} de {n_itens:,}".replace(",", "."))
print(f"Receita total considerada: R$ {br(base['price'].sum())}")
display(receita_categoria.head(10))

Itens considerados: 112.101 de 112.650
Receita total considerada: R$ 13.494.400,74


,categoria,pedidos,itens_vendidos,receita
0,health_beauty,8800,9634,1255695.13
1,watches_gifts,5604,5970,1198185.21
2,bed_bath_table,9399,11097,1035964.06
3,sports_leisure,7673,8590,979740.92
4,computers_accessories,6654,7781,904322.02
5,furniture_decor,6425,8298,727465.05
6,housewares,5847,6915,626825.80
7,cool_stuff,3616,3779,620770.49
8,auto,3872,4204,586585.73
9,garden_tools,3505,4328,481009.94


## 4. P2 — Quais vendedores apresentam maior volume de vendas?
**Arquivos:** pedidos, itens e vendedores. O volume é medido por receita, quantidade de itens e quantidade de pedidos distintos.

In [4]:
ranking_vendedores = (base.groupby("seller_id")
                          .agg(pedidos=("order_id", "nunique"),
                               itens_vendidos=("order_id", "size"),
                               receita=("price", "sum"))
                          .round(2).sort_values("receita", ascending=False).reset_index()
                          .merge(vendedores, on="seller_id", how="left"))

top10_vendedores = ranking_vendedores.head(10)
vendedores_sp_top10 = int((top10_vendedores["seller_state"] == "SP").sum())

print(f"Vendedores com pelo menos uma venda considerada: {len(ranking_vendedores):,}".replace(",", "."))
print(f"Dos 10 maiores vendedores, {vendedores_sp_top10} estão em SP.")
display(top10_vendedores[["seller_id", "seller_city", "seller_state", "pedidos", "itens_vendidos", "receita"]])

Vendedores com pelo menos uma venda considerada: 3.053
Dos 10 maiores vendedores, 9 estão em SP.


,seller_id,seller_city,seller_state,pedidos,itens_vendidos,receita
0,4869f7a5dfa277a7dca6462dcf3b52b2,guariba,SP,1131,1155,229237.63
1,53243585a1d6dc2643021fd1853d8905,lauro de freitas,BA,358,410,222776.05
2,4a3ca9315b744ce9f8e9374361493884,ibitinga,SP,1804,1985,200326.12
3,fa1c13f2614d7b5c4749cbc52fecda94,sumare,SP,584,585,192842.13
4,7c67e1448b00f6e969d365cea6b010ab,itaquaquecetuba,SP,982,1364,187923.89
5,7e93a43ef30c4f03f38b393420bc753a,barueri,SP,332,336,172583.88
6,da8622b14eb17ae2831f4ac5b9dab84a,piracicaba,SP,1314,1551,160236.57
7,7a67c85e85bb2ce8582c35f2203ad736,sao paulo,SP,1159,1170,141715.54
8,1025f0e2d44d7041d6cf58b6550e0bfa,sao paulo,SP,915,1428,138968.55
9,955fee9216a65b617aa5c0531780ce60,sao paulo,SP,1286,1498,135159.70


## 5. P3 — Como os pedidos se distribuem por estado?
**Arquivos:** pedidos e clientes. O estado é o do **cliente** (`customer_state`). Aqui contamos todos os pedidos, de qualquer status.

In [5]:
n_pedidos = len(pedidos)
pedidos_estado = pedidos.merge(clientes[["customer_id", "customer_state"]], on="customer_id", how="left")
assert len(pedidos_estado) == n_pedidos, "A junção duplicou ou perdeu linhas de pedidos"

distribuicao_estado = (pedidos_estado["customer_state"].value_counts()
                       .rename_axis("UF").reset_index(name="pedidos"))
distribuicao_estado["percentual"] = (100 * distribuicao_estado["pedidos"]
                                     / distribuicao_estado["pedidos"].sum()).round(2)

print(f"Pedidos: {n_pedidos:,} | Estados: {len(distribuicao_estado)}".replace(",", "."))
display(distribuicao_estado.head(10))

Pedidos: 99.441 | Estados: 27


,UF,pedidos,percentual
0,SP,41746,41.98
1,RJ,12852,12.92
2,MG,11635,11.70
3,RS,5466,5.50
4,PR,5045,5.07
5,SC,3637,3.66
6,BA,3380,3.40
7,DF,2140,2.15
8,ES,2033,2.04
9,GO,2020,2.03


## 6. P4 — Qual é o tempo médio de entrega dos pedidos?
**Arquivos:** pedidos (e clientes, para detalhar por estado).  
Tempo de entrega = data de entrega ao cliente menos data da compra, em dias. O prazo estimado é a data estimada menos a data da compra.

In [6]:
entregues = pedidos[(pedidos["order_status"] == "delivered")
                    & pedidos["order_delivered_customer_date"].notna()].copy()
entregues["dias_entrega"] = ((entregues["order_delivered_customer_date"]
                              - entregues["order_purchase_timestamp"]).dt.total_seconds() / SEGUNDOS_POR_DIA)
entregues["dias_prazo_estimado"] = ((entregues["order_estimated_delivery_date"]
                                     - entregues["order_purchase_timestamp"]).dt.total_seconds() / SEGUNDOS_POR_DIA)

media_entrega = entregues["dias_entrega"].mean()
media_prazo = entregues["dias_prazo_estimado"].mean()

print(f"Pedidos entregues com data de entrega: {len(entregues):,}".replace(",", "."))
print(f"Tempo médio até a entrega: {br(media_entrega, 1)} dias")
print(f"Prazo médio estimado: {br(media_prazo, 1)} dias")

entregues_estado = entregues.merge(clientes[["customer_id", "customer_state"]], on="customer_id", how="left")
assert len(entregues_estado) == len(entregues)
entrega_por_estado = (entregues_estado.groupby("customer_state")
                      .agg(pedidos=("order_id", "size"), dias_medios=("dias_entrega", "mean"))
                      .round(1).sort_values("dias_medios").reset_index())

print("\nEstados com menor tempo médio de entrega:")
display(entrega_por_estado.head(5))
print("Estados com maior tempo médio de entrega:")
display(entrega_por_estado.tail(5).iloc[::-1])

Pedidos entregues com data de entrega: 96.470
Tempo médio até a entrega: 12,6 dias
Prazo médio estimado: 23,7 dias

Estados com menor tempo médio de entrega:


,customer_state,pedidos,dias_medios
0,SP,40494,8.8
1,MG,11354,12.0
2,PR,4923,12.0
3,DF,2080,13.0
4,SC,3546,15.0


Estados com maior tempo médio de entrega:


,customer_state,pedidos,dias_medios
26,RR,41,29.4
25,AP,67,27.2
24,AM,145,26.4
23,AL,397,24.5
22,PA,946,23.8


## 7. P5 — Existe relação entre atrasos nas entregas e avaliações dos clientes?
**Arquivos:** pedidos e avaliações.  
Como alguns pedidos têm mais de uma avaliação, calculamos a **nota média por pedido** e só então comparamos os grupos. Atraso = entrega ao cliente depois da data estimada. A análise considera pedidos entregues que têm avaliação.

In [7]:
nota_pedido = avaliacoes.groupby("order_id")["review_score"].mean().rename("nota").reset_index()

analise_atraso = entregues.merge(nota_pedido, on="order_id", how="inner")
analise_atraso["dias_atraso"] = ((analise_atraso["order_delivered_customer_date"]
                                  - analise_atraso["order_estimated_delivery_date"]).dt.total_seconds()
                                 / SEGUNDOS_POR_DIA)
analise_atraso["situacao"] = analise_atraso["dias_atraso"].gt(0).map({True: "Atrasado", False: "No prazo"})

por_situacao = (analise_atraso.groupby("situacao")
                .agg(pedidos=("order_id", "size"), nota_media=("nota", "mean"))
                .round(2).reset_index())

faixas = ["No prazo", "1 a 3 dias", "4 a 7 dias", "Mais de 7 dias"]
analise_atraso["faixa_atraso"] = pd.cut(analise_atraso["dias_atraso"],
                                        bins=[-float("inf"), 0, 3, 7, float("inf")], labels=faixas)
por_faixa = (analise_atraso.groupby("faixa_atraso", observed=True)
             .agg(pedidos=("order_id", "size"), nota_media=("nota", "mean"))
             .round(2).reset_index())

print("Nota média por situação da entrega:")
display(por_situacao)
print("Nota média por faixa de atraso:")
display(por_faixa)

Nota média por situação da entrega:


,situacao,pedidos,nota_media
0,Atrasado,7661,2.57
1,No prazo,88163,4.29


Nota média por faixa de atraso:


,faixa_atraso,pedidos,nota_media
0,No prazo,88163,4.29
1,1 a 3 dias,2636,3.77
2,4 a 7 dias,1773,2.32
3,Mais de 7 dias,3252,1.73


## 8. Verificações de cuidado
Conferências que sustentam os cuidados descritos na seção 7 do documento: pedidos com mais de uma avaliação, diferença entre `customer_id` e `customer_unique_id` e volume de pedidos por mês (importante para séries temporais).

In [8]:
por_pedido = avaliacoes.groupby("order_id").size()
print("Pedidos com mais de uma avaliação:", int((por_pedido > 1).sum()))
print("Ocorrências excedentes de review_id:", int(avaliacoes["review_id"].duplicated().sum()))
print("customer_id distintos:", clientes["customer_id"].nunique())
print("customer_unique_id distintos (pessoas):", clientes["customer_unique_id"].nunique())

mensal = (pedidos.assign(mes=pedidos["order_purchase_timestamp"].dt.to_period("M"))
                 .groupby("mes").size().rename("pedidos").reset_index())
janela = mensal[(mensal["mes"] >= pd.Period("2017-01")) & (mensal["mes"] <= pd.Period("2018-08"))]
print("\nPedidos por mês fora da janela 01/2017–08/2018:")
display(mensal[(mensal["mes"] < pd.Period("2017-01")) | (mensal["mes"] > pd.Period("2018-08"))])
print(f"Dentro da janela 01/2017–08/2018: mínimo de {int(janela['pedidos'].min()):,} e "
      f"máximo de {int(janela['pedidos'].max()):,} pedidos por mês.".replace(",", "."))

Pedidos com mais de uma avaliação: 547
Ocorrências excedentes de review_id: 814
customer_id distintos: 99441
customer_unique_id distintos (pessoas): 96096

Pedidos por mês fora da janela 01/2017–08/2018:


,mes,pedidos
0,2016-09,4
1,2016-10,324
2,2016-12,1
23,2018-09,16
24,2018-10,4


Dentro da janela 01/2017–08/2018: mínimo de 800 e máximo de 7.544 pedidos por mês.


## 9. Resumo das evidências para a seção 7
A tabela abaixo é montada **a partir dos resultados calculados acima**, sem digitar números à mão. É o texto que alimenta a coluna "Evidência inicial" do documento.

In [9]:
lider = receita_categoria.iloc[0]
maior_vendedor = ranking_vendedores.iloc[0]
estados = distribuicao_estado.set_index("UF")["percentual"]
mais_rapido = entrega_por_estado.iloc[0]
mais_lento = entrega_por_estado.iloc[-1]
nota = por_situacao.set_index("situacao")["nota_media"]
nota_faixa = por_faixa.set_index("faixa_atraso")["nota_media"]

evidencias = pd.DataFrame([
    {"pergunta": "P1. Quais categorias de produtos geram mais receita?",
     "evidencia": f"A categoria {lider['categoria']} lidera, com R$ {br(lider['receita'] / 1_000_000)} milhão, "
                  f"seguida de {receita_categoria.iloc[1]['categoria']} e {receita_categoria.iloc[2]['categoria']}."},
    {"pergunta": "P2. Quais vendedores apresentam maior volume de vendas?",
     "evidencia": f"O maior vendedor soma R$ {br(maior_vendedor['receita'] / 1000, 0)} mil, "
                  f"e {vendedores_sp_top10} dos 10 maiores estão em SP."},
    {"pergunta": "P3. Como os pedidos se distribuem por estado?",
     "evidencia": f"{distribuicao_estado.iloc[0]['UF']} concentra {br(distribuicao_estado.iloc[0]['percentual'])}% dos pedidos, "
                  f"seguido de {distribuicao_estado.iloc[1]['UF']} ({br(distribuicao_estado.iloc[1]['percentual'])}%) "
                  f"e {distribuicao_estado.iloc[2]['UF']} ({br(distribuicao_estado.iloc[2]['percentual'])}%)."},
    {"pergunta": "P4. Qual é o tempo médio de entrega dos pedidos?",
     "evidencia": f"Média de {br(media_entrega, 1)} dias, contra {br(media_prazo, 1)} dias de prazo estimado; "
                  f"vai de {br(mais_rapido['dias_medios'], 1)} dias em {mais_rapido['customer_state']} "
                  f"a {br(mais_lento['dias_medios'], 1)} em {mais_lento['customer_state']}."},
    {"pergunta": "P5. Existe relação entre atrasos nas entregas e avaliações dos clientes?",
     "evidencia": f"Nota média de {br(nota['No prazo'])} nas entregas no prazo e de {br(nota['Atrasado'])} nas atrasadas; "
                  f"com mais de 7 dias de atraso, cai para {br(nota_faixa['Mais de 7 dias'])}."},
])
display(evidencias)

,pergunta,evidencia
0,P1. Quais categorias de produtos geram mais receita?,"A categoria health_beauty lidera, com R$ 1,26 milhão, seguida de watches_gifts e bed_bath_table."
1,P2. Quais vendedores apresentam maior volume de vendas?,"O maior vendedor soma R$ 229 mil, e 9 dos 10 maiores estão em SP."
2,P3. Como os pedidos se distribuem por estado?,"SP concentra 41,98% dos pedidos, seguido de RJ (12,92%) e MG (11,70%)."
3,P4. Qual é o tempo médio de entrega dos pedidos?,"Média de 12,6 dias, contra 23,7 dias de prazo estimado; vai de 8,8 dias em SP a 29,4 em RR."
4,P5. Existe relação entre atrasos nas entregas e avaliações dos clientes?,"Nota média de 4,29 nas entregas no prazo e de 2,57 nas atrasadas; com mais de 7 dias de atraso, ..."


## 10. Limitações e cuidados
- Os números são **evidências de viabilidade**, não conclusões de negócio. A receita considera só o preço dos itens e desconsidera frete, descontos e custos (a base não traz custo nem margem).
- A comparação entre atraso e nota mostra **associação**, não causa: outros fatores também influenciam a avaliação.
- A base é uma operação histórica (2016–2018) e anonimizada.
- Pedidos de 2016 e de setembro de 2018 em diante são poucos; séries mensais são mais confiáveis entre 01/2017 e 08/2018.
- O estado dos pedidos é o do cliente. Vendedores podem estar em outro estado.
- Regras de cálculo, tratamento de pedidos cancelados e modelagem definitiva devem ser validados em conjunto pela equipe.

## 11. Exportação do resumo (opcional)
O resumo é um arquivo pequeno, salvo em `docs/resultados_etapa1/`, para facilitar o preenchimento do documento da equipe. **Não são exportados os dados brutos.**

In [10]:
saida = RAIZ_PROJETO / "docs" / "resultados_etapa1"
saida.mkdir(parents=True, exist_ok=True)
evidencias.to_csv(saida / "evidencias_secao7.csv", index=False, encoding="utf-8-sig")
print("Resumo exportado para:", (saida / "evidencias_secao7.csv").resolve())

Resumo exportado para: C:\Users\Lenovo\Downloads\analisededados\gestao-e-analise-ecommerce\docs\resultados_etapa1\evidencias_secao7.csv
